In [4]:
    #import libraries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


In [5]:
#1. import database

In [6]:
conn = sqlite3.connect('customer_churn.db')

sql_query = """
     SELECT name
     FROM sqlite_master
     WHERE type='table';
"""
table = pd.read_sql(sql_query, conn)

for table_name in table['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"created dataframe: df_{table_name}")

conn.close()


created dataframe: df_db_customer
created dataframe: df_db_subscription
created dataframe: df_db_support


In [7]:
conn = sqlite3.connect('customer_churn.db')

for table_name in table['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [8]:
#data cleaning

In [9]:
df_db_customer.head()
df_db_customer.tail()


,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


In [10]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [11]:
#a.remae col-name
#b.drop column interest and pincode
#c. change dob datatype
#d.data standardization -gender
#e.fix missing value -country


In [12]:
#a.remae col-name
df_db_customer.rename(columns={'name' : 'customer_name'}, inplace= True)


In [13]:
#b. drop columns - interest and pincode
#df_db_customer.drop(df_db_customer.columns[-2:], axis=1)
#df_db_customer.drop(df_db_customer.columns[6:], axis=1)

df_db_customer.drop(columns=['interest', 'pincode'], inplace=True)

KeyError: "['interest'] not found in axis"

In [ ]:
df_db_customer.info()

In [ ]:
#c. change dob datatype
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])
               

In [ ]:
#d.data standardization -gender
#df_db_customer['gender'].unique()
df_db_customer['gender'].replace({'Men' : 'Male' , 'Women' : 'Female'})

In [14]:
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female', 'Women', 'Men']
Length: 4, dtype: str

In [15]:

#e.fix missing value -country
df_db_customer[df_db_customer['country'].isna()]


,customerid,customer_name,country,state,gender,dob,interests,pincode
5,0013-MHZWF,durga,NaN,Delhi,Women,1988-12-10 00:00:00,NaN,None
8,0015-UOCOJ,maya,NaN,Kathmandu,Women,1985-07-07 00:00:00,NaN,None
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01 00:00:00,NaN,None


In [16]:
# country and state - unique value pair
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

In [17]:
 df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [18]:
df_db_customer[df_db_customer['country'].isna()]


,customerid,customer_name,country,state,gender,dob,interests,pincode


In [19]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [20]:
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


In [21]:
date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']

df_db_subscription[date_col] = df_db_subscription [date_col].apply(pd.to_datetime)

In [22]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,NaN


In [23]:
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


In [24]:
df_db_support.drop(columns=['col_1', 'comment'], inplace=True)

In [25]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

In [26]:
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 420.0 bytes


In [27]:
# feature engineering and data analysis
#create a new col using existing col - churn flag 
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)

In [28]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88,1


In [29]:
# first fix support table duplicates then merge
df = (df_db_subscription
    .merge(df_db_customer, on = 'customerid' ,how = 'left')
    .merge(df_db_support, on = 'customerid' ,how = 'left'))

In [30]:
df_db_subscription.shape

(21, 12)

In [31]:
df.shape

(23, 22)

In [32]:
df.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,customer_name,country,state,gender,dob,interests,pincode,complaint_date,escalations,csat_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None,NaT,NaN,NaN
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None,2024-08-28,N,60.0
2,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None,2024-08-28,Y,10.0
3,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None,NaT,NaN,NaN
4,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None,NaT,NaN,NaN


In [33]:
df_db_subscription['customerid'].nunique()

21

In [34]:
df_db_customer['customerid'].nunique()

21

In [35]:
df_db_support['customerid'].size

9

In [36]:
df_db_support

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30
5,0017-IUDMW,2024-04-10,Y,25
6,0019-EFAEP,2024-09-27,Y,30
7,0022-TCJCI,2024-09-13,Y,10
8,0022-TCJCI,2024-09-14,N,90


In [37]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [38]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep= 'last')

In [39]:
df_db_support['customerid'].size

7

In [40]:
# merge df
df = (df_db_subscription
    .merge(df_db_customer, on = 'customerid' ,how = 'left')
    .merge(df_db_support, on = 'customerid' ,how = 'left'))

In [41]:
df.shape

(21, 23)

In [42]:
df.to_csv('exported_churn_data.csv', index=False)

In [43]:
#Data analysis
#1.Churn Rate 
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'interests', 'pincode', 'complaint_date', 'escalations', 'csat_score',
       'complaint_count'],
      dtype='str')

In [44]:
#df_db_subscription['churn_flag'].mean()*100
#print("churn Rate = ", round(churn_rate,2),"%") 
#df['churn_flag'].mean()

churn_rate = df_db_subscription['churn_flag'].mean() * 100

print("Churn rate =", round(churn_rate, 2), "%")

Churn rate = 28.57 %


In [45]:
#2.Retenion rate
retention_rate = 100 - churn_rate
print("Retention rate =", round(retention_rate, 2), "%")


Retention rate = 71.43 %


In [46]:
#churn by plan type
#df.groupby('plan_type')['churn_flag'].mean()
# Churn by plan type
#churn_by_plan = df_db_subscription.groupby('plan_type')['churn_flag'].mean().reset_index().mul(100).round(2).reset_index(name='churn_rate_pct')

#print("churn by plan =", churn_by_plan)
churn_by_plan = (
    df_db_subscription.groupby('plan_type')['churn_flag']
    .mean() * 100
)

churn_by_plan = churn_by_plan.round(2).reset_index()

churn_by_plan.columns = ['plan_type', 'churn_rate_pct']

print(churn_by_plan)

  plan_type  churn_rate_pct
0     Basic           60.00
1   Premium           14.29
2  Standard           22.22


In [47]:

#4.a.churn by plan type + sum (revenue) & count of users
#4. b.churn by subscription type
#calculate customer age 

In [48]:
#5. ARPU-AVG Revenue per user
arpu = df['monthly_charges'].mean()
print('ARPU = ', round(arpu,2))

ARPU =  18.85


In [49]:
#6.Avg. customer tenure
# count of days users has used our service : cancellation date else current date
today = pd.Timestamp.today().normalize()

df['tenure_days'] = (
    today - df['subscription_start_date']
).dt.days

# Calculate average tenure
avg_tenure = df['tenure_days'].mean()

print("Avg Tenure (Days) =", round (avg_tenure),0)

Avg Tenure (Days) = 1749 0


In [50]:
#7. revenue at risk  - revenue lost from churned users
revenue_at_risk = df.loc[
    df['churn_flag'] == 1,
    'monthly_charges'
].sum()

print("Revenue at risk (Rs_K) =", revenue_at_risk)

Revenue at risk (Rs_K) = 73.94


In [51]:
#8. Esclation Rate 
#escalation_rate = (df['escalations']==).mean()*100
#print("Esclation Rate =", round(escalation_rate,2),"%")

escalation_rate = (df['escalations'] == 1).mean() * 100
print("Escalation Rate =", round(escalation_rate, 2), "%")

Escalation Rate = 0.0 %


In [55]:
#9. Avg complaint per user 
avg_complaint = df['complaint_count'].sum() / df['customerid'].nunique() 
print("avg complaint per user = ", round(avg_complaint,2))

avg complaint per user =  0.43


In [59]:
#10.coorelation Esclation vs churn
df[['escalations', 'churn_flag']].dropna()

,escalations,churn_flag
1,Y,1
4,Y,1
5,N,0
6,N,1
11,Y,1
13,Y,1
18,N,1


In [52]:

df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'interests', 'pincode', 'complaint_date', 'escalations', 'csat_score',
       'complaint_count', 'tenure_days'],
      dtype='str')

In [56]:
df.head(2)

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,state,gender,dob,interests,pincode,complaint_date,escalations,csat_score,complaint_count,tenure_days
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,Maharashtra,Male,1982-04-12 00:00:00,travel,None,NaT,NaN,NaN,NaN,1999
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,Karnataka,Male,1995-11-23 00:00:00,NaN,None,2024-08-28,Y,10.0,2.0,2225
